# Gold Layer - Repository Health Metrics

This notebook aggregates repository-level metrics from the Silver layer to calculate the Repository Health Score.

**Objectives**
- Aggregate pull request, issue and contributor statistics.
- Join repository metadata with operational metrics.
- Calculate a normalized Repository Health Score (0–100).
- Generate repository-level KPIs for dashboards and repository comparison.
- Add Gold layer processing metadata.

In [0]:
from pyspark.sql import functions as F

repo = spark.table("gitobservatory.silver.repositories")
pr = spark.table("gitobservatory.silver.pull_requests")
issue = spark.table("gitobservatory.silver.issues")
contributor = spark.table("gitobservatory.silver.contributors")

pr_metrics = pr.groupBy("repo_id","repo_full_name").agg(
    F.count("*").alias("total_pull_requests"),
    F.sum(F.when(F.col("merged"), 1).otherwise(0)).alias("merged_pull_requests"),
    F.sum(F.when(F.col("state") == "open", 1).otherwise(0)).alias("open_pull_requests"),
    F.sum(F.when(F.col("draft"), 1).otherwise(0)).alias("draft_pull_requests"))
issue_metrics = issue.groupBy("repo_id","repo_full_name").agg(
    F.count("*").alias("total_issues"),
    F.sum(F.when(F.col("state") == "open", 1).otherwise(0)).alias("open_issues_count"),
    F.sum(F.when(F.col("state") == "closed", 1).otherwise(0)).alias("closed_issues_count"))

contributor_metrics = contributor.groupBy("repo_id","repo_full_name").agg(F.count("*").alias("total_contributors"))

gold_repo = (
    repo
    .join(pr_metrics.drop("repo_full_name"),"repo_id","left")
    .join(issue_metrics.drop("repo_full_name"),"repo_id","left")
    .join(contributor_metrics.drop("repo_full_name"),"repo_id","left"))

gold_repo = gold_repo.fillna(0)

gold_repo = gold_repo.withColumn("star_score",F.least(F.col("stars") / 10000, F.lit(1.0)) * 30)

gold_repo = gold_repo.withColumn("fork_score",F.least(F.col("forks") / 2000, F.lit(1.0)) * 15)

gold_repo = gold_repo.withColumn("contributor_score",F.least(F.col("total_contributors") / 500, F.lit(1.0)) * 15)

gold_repo = gold_repo.withColumn("pr_score",F.when(F.col("total_pull_requests") > 0,(F.col("merged_pull_requests")/ F.col("total_pull_requests")) * 20).otherwise(0))
gold_repo = gold_repo.withColumn("issue_score",F.when(F.col("total_issues") > 0,(F.col("closed_issues_count")/ F.col("total_issues")) * 20).otherwise(0))

gold_repo = gold_repo.withColumn("repository_health_score",F.round(F.col("star_score")+ F.col("fork_score")+ F.col("contributor_score")+ F.col("pr_score")+ F.col("issue_score"),2))

gold_repo = gold_repo.withColumn("gold_processed_timestamp",F.current_timestamp())
gold_repo.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("gitobservatory.gold.repository_health")


In [0]:
%sql
SELECT * FROM gitobservatory.gold.repository_health;

repo_id,owner,repository_name,repo_full_name,repository_key,source_system,bronze_loaded_timestamp,description,language,visibility,default_branch,homepage,license_name,created_at,updated_at,pushed_at,size,stars,forks,watchers,subscribers,open_issues,has_issues,has_projects,has_wiki,has_discussions,is_fork,is_archived,is_disabled,silver_processed_timestamp,total_pull_requests,merged_pull_requests,open_pull_requests,draft_pull_requests,total_issues,open_issues_count,closed_issues_count,total_contributors,star_score,fork_score,contributor_score,pr_score,issue_score,repository_health_score,gold_processed_timestamp
139199684,PrefectHQ,prefect,PrefectHQ/prefect,PrefectHQ/prefect,github,2026-07-01T16:07:12.374Z,Prefect is a workflow orchestration framework for building resilient data pipelines in Python.,Python,public,main,https://prefect.io,Apache License 2.0,2018-06-29T21:59:26.000Z,2026-07-01T15:31:13.000Z,2026-07-01T14:15:09.000Z,224988,22723,2356,22723,195,819,true,false,false,true,false,false,false,2026-07-01T17:00:50.045Z,5926,0,34,383,1782,391,1391,423,30.0,15.0,12.69,0.0,15.611672278338947,73.3,2026-07-01T17:01:44.415Z
53548867,dbt-labs,dbt-core,dbt-labs/dbt-core,dbt-labs/dbt-core,github,2026-07-01T16:07:10.813Z,dbt enables data analysts and engineers to transform their data using the same practices that software engineers use to build applications.,Rust,public,main,https://getdbt.com,Apache License 2.0,2016-03-10T02:38:00.000Z,2026-07-01T15:34:21.000Z,2026-07-01T16:04:20.000Z,84183,13332,2448,13332,160,1496,true,false,false,true,false,false,false,2026-07-01T17:00:50.045Z,1563,0,181,96,3232,1015,2217,376,30.0,15.0,11.28,0.0,13.719059405940595,70.0,2026-07-01T17:01:44.415Z


# Gold Layer - Pull Request Metrics

This notebook aggregates pull request information from the Silver layer into repository-level pull request metrics.

**Objectives**
- Calculate pull request activity by repository.
- Measure merge efficiency and pull request lifecycle metrics.
- Calculate average pull request size and review activity.
- Generate repository-level KPIs for dashboards and repository comparison.

In [0]:
from pyspark.sql import functions as F

pr = spark.table("gitobservatory.silver.pull_requests")

gold_pr = pr.groupBy(
    "repo_id",
    "repo_full_name"
).agg(
    F.count("*").alias("total_pull_requests"),
    F.sum(F.when(F.col("state") == "open", 1).otherwise(0)).alias("open_pull_requests"),
    F.sum(F.when(F.col("state") == "closed", 1).otherwise(0)).alias("closed_pull_requests"),
    F.sum(F.when(F.col("merged"), 1).otherwise(0)).alias("merged_pull_requests"),

    F.sum(F.when(F.col("draft"), 1).otherwise(0)).alias("draft_pull_requests"),

    F.round(F.avg("merge_time_hours"),2).alias("avg_merge_time_hours"),
    F.round(F.avg("comments"),2).alias("avg_comments"),
    F.round(F.avg("review_comments"),2).alias("avg_review_comments"),
    F.round(F.avg("commits"),2).alias("avg_commits"),

    F.round(F.avg("changed_files"),2).alias("avg_changed_files"),
    F.round(F.avg("additions"),2).alias("avg_additions"),
    F.round(F.avg("deletions"),2).alias("avg_deletions"))

gold_pr = gold_pr.withColumn("merge_rate",F.when(F.col("total_pull_requests") > 0,F.round((F.col("merged_pull_requests")/ F.col("total_pull_requests")) * 100,2)).otherwise(0))
gold_pr = gold_pr.withColumn("merge_efficiency_score",F.round((F.col("merge_rate")* (100 - F.least(F.col("avg_merge_time_hours"), F.lit(100)))) / 100,2))
gold_pr = gold_pr.withColumn("gold_processed_timestamp",F.current_timestamp())

gold_pr.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("gitobservatory.gold.pull_request_metrics")


# Gold Layer - Issue Metrics

This notebook aggregates issue information from the Silver layer into repository-level issue metrics.

**Objectives**
- Calculate issue activity by repository.
- Measure issue closure efficiency.
- Calculate issue resolution and issue age metrics.
- Measure issue management using labels, assignees and milestones.
- Generate repository-level KPIs for dashboards and repository comparison.

In [0]:
from pyspark.sql import functions as F

issue = spark.table("gitobservatory.silver.issues")

gold_issue = issue.groupBy("repo_id","repo_full_name").agg(
    F.count("*").alias("total_issues"),
    F.sum(F.when(F.col("state") == "open", 1).otherwise(0)).alias("open_issues"),
    F.sum(F.when(F.col("state") == "closed", 1).otherwise(0)).alias("closed_issues"),
    F.round(F.avg("issue_resolution_time_hours"),2).alias("avg_resolution_time_hours"),
    F.round(F.avg("comments"),2).alias("avg_comments"),
    F.round(F.avg("issue_age_days"),2).alias("avg_issue_age_days"),
    F.sum(F.when(F.col("has_labels"), 1).otherwise(0)).alias("issues_with_labels"),
    F.sum(F.when(F.col("has_assignees"), 1).otherwise(0)).alias("issues_with_assignees"),
    F.sum(F.when(F.col("has_milestone"), 1).otherwise(0)).alias("issues_with_milestones"))

gold_issue = gold_issue.withColumn("issue_closure_rate",F.when(F.col("total_issues") > 0,F.round((F.col("closed_issues")/ F.col("total_issues")) * 100,2)).otherwise(0))

gold_issue = gold_issue.withColumn("issue_management_score",F.round(((F.col("issues_with_labels")+ F.col("issues_with_assignees")+ F.col("issues_with_milestones"))/ (F.col("total_issues") * 3)) * 100,2))

gold_issue = gold_issue.withColumn("gold_processed_timestamp",F.current_timestamp())
gold_issue.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("gitobservatory.gold.issue_metrics")

# Gold Layer - Contributor Metrics

This notebook aggregates contributor information from the Silver layer into repository-level contributor metrics.

**Objectives**
- Calculate contributor statistics by repository.
- Classify contributors based on contribution activity.
- Measure contributor composition and engagement.
- Generate contributor-related KPIs for dashboards and repository comparison.

In [0]:
from pyspark.sql import functions as F
contributor = spark.table("gitobservatory.silver.contributors")
gold_contributor = contributor.groupBy(
    "repo_id",
    "repo_full_name"
).agg(
    F.count("*").alias("total_contributors"),
    F.sum(F.when(~F.col("is_bot"), 1).otherwise(0)).alias("human_contributors"),
    F.sum(F.when(F.col("is_bot"), 1).otherwise(0)).alias("bot_contributors"),
    F.sum(F.when(F.col("is_site_admin"), 1).otherwise(0)).alias("site_admins"),
    F.sum(F.when(F.col("is_top_contributor"), 1).otherwise(0)).alias("top_contributors"),
    F.round(F.avg("contributions"),2).alias("avg_contributions"),
    F.round(F.avg("contribution_score"),2).alias("avg_contribution_score"),
    F.sum(F.when(F.col("contribution_level") == "Expert", 1).otherwise(0)).alias("expert_contributors"),
    F.sum(F.when(F.col("contribution_level") == "Advanced", 1).otherwise(0)).alias("advanced_contributors"),

    F.sum(F.when(F.col("contribution_level") == "Intermediate", 1).otherwise(0)).alias("intermediate_contributors"),

    F.sum(F.when(F.col("contribution_level") == "Beginner", 1).otherwise(0)).alias("beginner_contributors"),
    F.sum(F.when(F.col("profile_available"), 1).otherwise(0)).alias("profiles_available"))

gold_contributor = gold_contributor.withColumn("top_contributor_percentage",F.when(F.col("total_contributors") > 0,F.round((F.col("top_contributors")/ F.col("total_contributors")) * 100,2)).otherwise(0))

gold_contributor = gold_contributor.withColumn("gold_processed_timestamp",F.current_timestamp())
gold_contributor.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("gitobservatory.gold.contributor_metrics")

# Gold Layer - Pull Request Review Metrics

This notebook aggregates pull request review information from the Silver layer into repository-level review metrics.

**Objectives**
- Calculate review activity by repository.
- Measure review approval and change request rates.
- Calculate review coverage.
- Generate repository-level review KPIs for dashboards and repository comparison.

In [0]:
from pyspark.sql import functions as F

review = spark.table("gitobservatory.silver.pull_request_reviews")
pr = spark.table("gitobservatory.silver.pull_requests")

gold_review = (review.join(pr.select("pr_id", "repo_id", "repo_full_name"),"pr_id","left"))

gold_review = gold_review.groupBy("repo_id","repo_full_name").agg(
    F.countDistinct("review_id").alias("total_reviews"),
    F.countDistinct("pr_id").alias("reviewed_pull_requests"),
    F.sum(F.when(F.col("is_approved"),1).otherwise(0)).alias("approved_reviews"),
    F.sum(F.when(F.col("changes_requested"),1).otherwise(0)).alias("changes_requested_reviews"),
    F.sum(F.when(F.col("is_comment"),1).otherwise(0)).alias("comment_reviews"))
gold_review = gold_review.withColumn("approval_rate",F.when(F.col("total_reviews")>0,F.round(F.col("approved_reviews")/F.col("total_reviews")*100,2)).otherwise(0))

gold_review = gold_review.withColumn("change_request_rate",F.when(F.col("total_reviews")>0,F.round(F.col("changes_requested_reviews")/F.col("total_reviews")*100,2)).otherwise(0))

gold_review = gold_review.withColumn("gold_processed_timestamp",F.current_timestamp())
gold_review.write.format("delta").option("overwriteSchema","true").mode("overwrite").saveAsTable("gitobservatory.gold.review_metrics")

# Gold Layer - Workflow Metrics

This notebook aggregates GitHub Actions workflow execution data into repository-level CI/CD metrics.

**Objectives**
- Calculate workflow execution statistics.
- Measure workflow success and failure rates.
- Calculate average workflow execution time.
- Generate CI/CD KPIs for dashboards and repository comparison.

In [0]:
from pyspark.sql import functions as F

workflow = spark.table("gitobservatory.silver.workflow_runs")

gold_workflow = workflow.groupBy("repo_id","repo_full_name").agg(
    F.count("*").alias("total_workflow_runs"),
    F.sum(F.when(F.col("is_success"),1).otherwise(0)).alias("successful_runs"),
    F.sum(F.when(F.col("is_failure"),1).otherwise(0)).alias("failed_runs"),
    F.sum(F.when(F.col("is_cancelled"),1).otherwise(0)).alias("cancelled_runs"),
    F.round(F.avg("workflow_duration_minutes"),2).alias("avg_workflow_duration_minutes"))

gold_workflow = gold_workflow.withColumn("workflow_success_rate",F.when(F.col("total_workflow_runs")>0,F.round(F.col("successful_runs")/F.col("total_workflow_runs")*100,2)).otherwise(0))
gold_workflow = gold_workflow.withColumn("gold_processed_timestamp",F.current_timestamp())
gold_workflow.write.format("delta").option("overwriteSchema","true").mode("overwrite").saveAsTable("gitobservatory.gold.workflow_metrics")

# Gold Layer - Repository Comparison

This notebook consolidates all Gold layer metrics into a unified repository comparison table.

**Objectives**
- Combine repository health, pull request, issue, contributor, review and workflow metrics.
- Calculate repository-level performance indicators.
- Generate an Overall Repository Score.
- Classify repositories into health grades.
- Produce a single comparison table for the GitObservatory web application.

In [0]:
from pyspark.sql import functions as F


repo = spark.table("gitobservatory.gold.repository_health")

pr = spark.table("gitobservatory.gold.pull_request_metrics").select(
    "repo_id","merge_rate","avg_merge_time_hours","avg_comments","avg_review_comments","avg_commits","avg_changed_files", "avg_additions","avg_deletions","merge_efficiency_score")

issue = spark.table("gitobservatory.gold.issue_metrics").select("repo_id","issue_closure_rate","avg_resolution_time_hours","avg_issue_age_days","issue_management_score")

contributor = spark.table("gitobservatory.gold.contributor_metrics").select("repo_id","avg_contributions","avg_contribution_score","top_contributor_percentage")

review = spark.table("gitobservatory.gold.review_metrics").select("repo_id","approval_rate","change_request_rate","total_reviews")
workflow = spark.table("gitobservatory.gold.workflow_metrics").select("repo_id","workflow_success_rate","avg_workflow_duration_minutes")

comparison = (
    repo
    .join(pr, "repo_id", "left")
    .join(issue, "repo_id", "left")
    .join(contributor, "repo_id", "left")
    .join(review, "repo_id", "left")
    .join(workflow, "repo_id", "left")
)
comparison = comparison.fillna(0)

comparison = comparison.withColumn("overall_repository_score",F.round((F.col("repository_health_score") * 0.30) +(F.col("merge_rate") * 0.20) +(F.col("issue_closure_rate") * 0.15) +(F.col("workflow_success_rate") * 0.15) +(F.col("approval_rate") * 0.10) + (F.col("avg_contribution_score") * 25 * 0.10),2))

comparison = comparison.withColumn(
    "repository_grade",
    F.when(F.col("repository_health_score") >= 90, "A+")
     .when(F.col("repository_health_score") >= 80, "A")
     .when(F.col("repository_health_score") >= 70, "B")
     .when(F.col("repository_health_score") >= 60, "C")
     .otherwise("D")
)

comparison = comparison.withColumn(
    "repository_status",
    F.when(F.col("repository_health_score") >= 90, "Excellent")
     .when(F.col("repository_health_score") >= 70, "Healthy")
     .when(F.col("repository_health_score") >= 50, "Moderate")
     .otherwise("Needs Improvement")
)

comparison = comparison.select(
    "repo_id",
    "repo_full_name",
    "repository_key",
    "stars",
    "forks",
    "watchers",
    "subscribers",
    "repository_health_score",
    "overall_repository_score",
    "repository_grade",
    "repository_status",
    "total_pull_requests",
    "merged_pull_requests",
    "merge_rate",
    "merge_efficiency_score",
    "avg_merge_time_hours",
    "total_issues",
    "closed_issues_count",
    "issue_closure_rate",
    "issue_management_score",
    "avg_resolution_time_hours",
    "total_contributors",
    "avg_contributions",
    "avg_contribution_score",
    "top_contributor_percentage",
    "approval_rate",
    "change_request_rate",
    "total_reviews",
    "workflow_success_rate",
    "avg_workflow_duration_minutes",
    "gold_processed_timestamp")

comparison.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("gitobservatory.gold.repository_comparison")
